# Coursera
Creacion del conjunto de datos a partir del json que surge del web scrapping. Paso 4: recomendar cursos a partir de skills de coursera.

In [21]:
import pandas as pd

# Cargar directamente desde archivo JSON
df_final = pd.read_json("courses.json")

display(df_final.head())


,url,name,topics,skills,description,language,outcomes
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]","[Application Security, threat intelligence, ne...",This course gives you the background needed to...,English,[]
1,https://www.coursera.org/learn/-network-security,Network Security | Coursera,"[Information Technology, Security]","[Cyberattacks, Network Security, Threat]",Welcome to course 4 of 5 of this Specializatio...,English,[]
2,https://www.coursera.org/learn/-security-princ...,Security Principles | Coursera,"[Information Technology, Security]","[Risk, Information Assurance, security, govern...",Welcome to course 1 of 5 of this Specializatio...,English,[]
3,https://www.coursera.org/learn/10k-women-1,"Grow Your Business with Goldman Sachs 10,000 W...","[Business, Entrepreneurship]","[Opportunity Identification, Strategic Thinkin...",This free online course is one of 10 courses a...,English,[Develop your community and discuss the value ...
4,https://www.coursera.org/learn/10k-women-10,"Fundamentals of Negotiation, with Goldman Sach...","[Business, Entrepreneurship]","[Emotional Intelligence, Customer Relationship...",This free online course is one of 10 courses a...,English,"[Review how to establish rapport, trust, and r..."


## Arreglo de skills

In [22]:
df_exploded = df_final.explode("skills")
display(df_exploded)

,url,name,topics,skills,description,language,outcomes
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",Application Security,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",threat intelligence,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",network defensive tactics,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",security analyst,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",Cybersecurity,This course gives you the background needed to...,English,[]
...,...,...,...,...,...,...,...
5263,https://www.coursera.org/learn/zn-and-ni-based...,Zn and Ni Based Batteries | Coursera,"[Physical Science and Engineering, Electrical ...",battery charging,Zn and Ni based Batteries: This course focuses...,English,"[ Participants will learn active materials, ch..."
5264,https://www.coursera.org/learn/zos-rexx-progra...,IBM z/OS Rexx Programming | Coursera,"[Computer Science, Software Development]",Computer Programming,This course is designed to teach you the basic...,English,"[Write programs using REXX, Create user-define..."
5264,https://www.coursera.org/learn/zos-rexx-progra...,IBM z/OS Rexx Programming | Coursera,"[Computer Science, Software Development]",Mainframe Coding,This course is designed to teach you the basic...,English,"[Write programs using REXX, Create user-define..."
5264,https://www.coursera.org/learn/zos-rexx-progra...,IBM z/OS Rexx Programming | Coursera,"[Computer Science, Software Development]",REXX,This course is designed to teach you the basic...,English,"[Write programs using REXX, Create user-define..."


In [23]:
# Eliminar filas donde 'skills' esté vacío o NaN
df_exploded = df_exploded[df_exploded["skills"].notna() & (df_exploded["skills"] != "")]


In [24]:
import re, unicodedata

# --- Regex precompiles ---
RE_ZERO_WIDTH   = re.compile(r'[\u200B-\u200F\u202A-\u202E\u2060\uFEFF]')
RE_SPACES       = re.compile(r'\s+')
RE_TRAILING_DOT = re.compile(r'[.\s]+$')

# Clase de comillas (ASCII + tipográficas + angulares)
QUOTE_UTF8_CHARS = '"\'`“”„‟‘’‚‛«»‹›'
RE_EDGE_QUOTES   = re.compile(rf'^[{re.escape(QUOTE_UTF8_CHARS)}\s]+|[{re.escape(QUOTE_UTF8_CHARS)}\s]+$')

# Mojibake -> UTF-8/ASCII
MOJIBAKE_MAP = {
    "â¢": "•",
    "â": "—",
    "â": "–",
    "â¦": "…",
    "â": "'",
    "â": "'",
    "â": '"',
    "â": '"',
    "â": "",
}

# Enumeraciones iniciales (NO metas comillas aquí)
LEADING_ENUM_PATTERNS = [
    re.compile(r'^\s*\d+\s*\t+\s*'),               # "12\t..."
    re.compile(r'^\s*\d+\s*[\.\)]\s*'),            # "1." / "2)"
    re.compile(r'^\s*[\(\[]\s*\d+\s*[\)\]]\s*'),   # "(1)" / "[2]"
    re.compile(r'^\s*\d+\s*[-–—]\s+'),             # "1 - " / "1 – " / "1 — "
    re.compile(r'^\s*[A-Za-z]\s*[\.\)]\s+'),       # "a) " / "B. "
    re.compile(r'^\s*[ivxlcdmIVXLCDM]+\s*[\.\)]\s+'),  # "IV. " / "i) "
    re.compile(r'^\s*[•\-\*\u2219·–—]\s*\t+\s*'),  # bullets con tab
    re.compile(r'^\s*[•\-\*\u2219·–—]\s+'),        # bullets con espacio
]

def _strip_edge_quotes_loop(s: str) -> str:
    """Recorta comillas/espacios en bordes de forma repetida (izq./der.)."""
    while True:
        new_s = RE_EDGE_QUOTES.sub('', s).strip()
        if new_s == s:
            return s
        s = new_s

def clean_skill(text: str) -> str:
    """
    Devuelve la skill limpia en minúsculas:
      - quita enumeraciones iniciales,
      - recorta comillas de borde (ASCII/UTF-8),
      - corrige mojibake, elimina zero-width/bidi/BOM,
      - quita punto(s) finales,
      - conserva paréntesis internos.
    """
    if text is None:
        return ""
    s = unicodedata.normalize("NFKC", str(text))

    # Mojibake -> UTF8/ASCII
    for bad, good in MOJIBAKE_MAP.items():
        if bad in s:
            s = s.replace(bad, good)

    # Invisibles
    s = RE_ZERO_WIDTH.sub("", s)

    # Enumeraciones al inicio (pelar en capas)
    prev = None
    while s != prev:
        prev = s
        for pat in LEADING_ENUM_PATTERNS:
            s = pat.sub("", s, count=1)

    # Normaliza espacios
    s = RE_SPACES.sub(" ", s).strip()

    # 1ª pasada: recortar comillas de borde
    s = _strip_edge_quotes_loop(s)

    # Quitar punto(s) finales
    s = RE_TRAILING_DOT.sub("", s).strip()

    # 2ª pasada: recortar comillas de borde (para casos como ”.)
    s = _strip_edge_quotes_loop(s)

    # Minúsculas
    return s.lower().strip()

def _t(inp, expected):
    got = clean_skill(inp)
    assert got == expected, f"\nINPUT : {repr(inp)}\nGOT   : {repr(got)}\nEXPECT: {repr(expected)}"

# Comillas (ASCII, tipográficas, angulares) + punto final
_t('"Python Programming"', 'python programming')
_t('“Python Programming”.', 'python programming')
_t("‘Skill’", 'skill')
_t("«Skill»", 'skill')
_t("‹Skill›", 'skill')
_t("`Skill`", 'skill')
_t("' Skill '", 'skill')

# Punto final en skills largas
_t("Long descriptive skill.", "long descriptive skill")
_t("C++.", "c++")

# Mayúsculas/minúsculas
_t("JAVA", "java")
_t("Title Case Mixed", "title case mixed")

# Enumeraciones (número + tab / . / ) / (1) / [2] / 1 - / letra) / romanos / bullets
_t("12\tSystems Design", "systems design")
_t("1.\tSkill", "skill")
_t("(1)\tSkill", "skill")
_t("[2] Skill", "skill")
_t("1 - Skill", "skill")
_t("a)\tSkill", "skill")
_t("IV.\tSkill", "skill")
_t("•\tLeadership", "leadership")
_t("-\tProject Management", "project management")
_t("—\tSystems", "systems")
_t("–\tSystems", "systems")
_t("• bullet style skill", "bullet style skill")
_t("·\tAnother Skill", "another skill")

# Mojibake en medio del texto + zero-width
_t("4Aâs Marketing", "4a's marketing")
_t("Hâow to Design", "how to design")

# Paréntesis preservados
_t("Design (CAD)", "design (cad)")
_t("A/B Testing (advanced)", "a/b testing (advanced)")


In [25]:
temp=df_exploded['skills'].apply(clean_skill)


display(temp)

0            application security
0             threat intelligence
0       network defensive tactics
0                security analyst
0                   cybersecurity
                  ...            
5263                battery types
5263             battery charging
5264         computer programming
5264             mainframe coding
5264                         rexx
Name: skills, Length: 12674, dtype: object

In [26]:
conteo = pd.DataFrame(temp.value_counts())

display(conteo)

,count
skills,
data analysis,170
python programming,160
machine learning,117
communication,106
data visualization (dataviz),83
...,...
triboelectric,1
designing and redesigning energy systems to put justice at the center,1
recognizing energy injustice,1


In [27]:
for i in temp:
    print(i) 


application security
threat intelligence
network defensive tactics
security analyst
cybersecurity
cyberattacks
network security
threat
risk
information assurance
security
governance
opportunity identification
strategic thinking
business analytics
entrepreneurship
emotional intelligence
customer relationship management (crm)
communication
bargaining
negotiation
income statement analysis
balance sheet
financial terminology
financial statement use and analysis
financial ratio analysis
leadership
strategic planning
vision statement development
business culture analysis
customer value proposition (cvp) development
strategic pricing
competitive advantage analysis
competitive analysis
financial management
strategic planning
cash flow cycle analysis
cash flow forecast analysis
market research
marketing strategy development
value proposition analysis
brand identity and management
process management
operations management
business modelling
communication
leadership
performance management
business

In [28]:
df_exploded2 = df_exploded.copy()
df_exploded2['skills'] = temp

display(df_exploded2)

,url,name,topics,skills,description,language,outcomes
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",application security,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",threat intelligence,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",network defensive tactics,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",security analyst,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",cybersecurity,This course gives you the background needed to...,English,[]
...,...,...,...,...,...,...,...
5263,https://www.coursera.org/learn/zn-and-ni-based...,Zn and Ni Based Batteries | Coursera,"[Physical Science and Engineering, Electrical ...",battery types,Zn and Ni based Batteries: This course focuses...,English,"[ Participants will learn active materials, ch..."
5263,https://www.coursera.org/learn/zn-and-ni-based...,Zn and Ni Based Batteries | Coursera,"[Physical Science and Engineering, Electrical ...",battery charging,Zn and Ni based Batteries: This course focuses...,English,"[ Participants will learn active materials, ch..."
5264,https://www.coursera.org/learn/zos-rexx-progra...,IBM z/OS Rexx Programming | Coursera,"[Computer Science, Software Development]",computer programming,This course is designed to teach you the basic...,English,"[Write programs using REXX, Create user-define..."
5264,https://www.coursera.org/learn/zos-rexx-progra...,IBM z/OS Rexx Programming | Coursera,"[Computer Science, Software Development]",mainframe coding,This course is designed to teach you the basic...,English,"[Write programs using REXX, Create user-define..."


In [38]:
df_exploded2 = df_exploded2.rename(columns={"skills": "skills_title"})
display(df_exploded2)

,url,name,topics,skills_title,description,language,outcomes
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",application security,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",threat intelligence,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",network defensive tactics,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",security analyst,This course gives you the background needed to...,English,[]
0,https://www.coursera.org/learn/ibm-cyber-threa...,Cyber Threat Intelligence | Coursera,"[Information Technology, Security]",cybersecurity,This course gives you the background needed to...,English,[]
...,...,...,...,...,...,...,...
5263,https://www.coursera.org/learn/zn-and-ni-based...,Zn and Ni Based Batteries | Coursera,"[Physical Science and Engineering, Electrical ...",battery types,Zn and Ni based Batteries: This course focuses...,English,"[ Participants will learn active materials, ch..."
5263,https://www.coursera.org/learn/zn-and-ni-based...,Zn and Ni Based Batteries | Coursera,"[Physical Science and Engineering, Electrical ...",battery charging,Zn and Ni based Batteries: This course focuses...,English,"[ Participants will learn active materials, ch..."
5264,https://www.coursera.org/learn/zos-rexx-progra...,IBM z/OS Rexx Programming | Coursera,"[Computer Science, Software Development]",computer programming,This course is designed to teach you the basic...,English,"[Write programs using REXX, Create user-define..."
5264,https://www.coursera.org/learn/zos-rexx-progra...,IBM z/OS Rexx Programming | Coursera,"[Computer Science, Software Development]",mainframe coding,This course is designed to teach you the basic...,English,"[Write programs using REXX, Create user-define..."


In [39]:
import csv
from pathlib import Path

formato =[".csv", ".tsv"] 

OUT_DIR2 = Path("results")
OUT_DIR2.mkdir(parents=True, exist_ok=True)

for i in formato:
    df_exploded2.to_csv(f"{OUT_DIR2}/courses_full{i}", index=False, encoding="utf-8",
                        quoting=csv.QUOTE_MINIMAL, quotechar='"', escapechar='\\')


## Skills

In [41]:
conteo = pd.DataFrame(df_exploded2["skills_title"].value_counts())

display(conteo)


,count
skills_title,
data analysis,170
python programming,160
machine learning,117
communication,106
data visualization (dataviz),83
...,...
triboelectric,1
designing and redesigning energy systems to put justice at the center,1
recognizing energy injustice,1


In [44]:
skills = pd.DataFrame(df_exploded2["skills_title"].unique(), 
                        columns=["skill_title"]
                        )
display(skills)

,skill_title
0,application security
1,threat intelligence
2,network defensive tactics
3,security analyst
4,cybersecurity
...,...
5299,panels
5300,tso
5301,sysop
5302,mainframe coding


In [ ]:
# skills = skills.rename(columns={"skills": "skills_title"})
# display(skills)

,0
0,application security
1,threat intelligence
2,network defensive tactics
3,security analyst
4,cybersecurity
...,...
5299,panels
5300,tso
5301,sysop
5302,mainframe coding


In [45]:
for i in formato:
    skills.to_csv(f"{OUT_DIR2}/courses_plain{i}", index=False, encoding="utf-8",
                        quoting=csv.QUOTE_MINIMAL, quotechar='"', escapechar='\\')
